In [ ]:
import boto3
from consts_api import *
from utils_api import get_model_response

In [ ]:
bedrock_client = boto3.client(service_name='bedrock-runtime', region_name="us-west-2")

In [ ]:
messages=[{
        "role": "user",
        "content": [{"text": "Multiply 1982982 by 2390831. Only respond with the result"}]
    }]

inference_config={"maxTokens":400}

# Send the message.
response = bedrock_client.converse(
    modelId=MODEL_ID,
    messages=messages,
    inferenceConfig=inference_config,
)

In [ ]:
get_model_response(response)

In [ ]:
def calculator(operation, operand1, operand2):
    if operation == "add":
        return operand1 + operand2
    elif operation == "subtract":
        return operand1 - operand2
    elif operation == "multiply":
        return operand1 * operand2
    elif operation == "divide":
        if operand2 == 0:
            raise ValueError("Cannot divide by zero.")
        return operand1 / operand2
    else:
        raise ValueError(f"Unsupported operation: {operation}")

In [ ]:
calculator("multiply",30,910)

## Tool Definition

In [ ]:
calculator_tool = {
    "toolSpec": {
        "name": "calculator",
        "description": "A simple calculator that performs basic arithmetic operations.",
        "inputSchema": {
            "json": {
                "type": "object",
                    "properties": {
                        "operation": {
                            "type": "string",
                            "enum": ["add","subtract","multiply","divide"],
                            "description": "The arithmetic operation to perform."
                        },
                        "operand1": {
                            "type": "number",
                            "description":"The first operand."
                        },
                        "operand2": {
                            "type": "number",
                            "description":"The second operand."
                        }
                    },
                    "required": ["operation", "operand1", "operand2"]
            }
        }
    }
}

## Providing Tools to Model

In [ ]:
messages=[{
        "role": "user",
        "content": [{"text": "Multiply 1982982 by 2390831. Only respond with the result"}]
    }]

inference_config={"maxTokens":400}

# Send the message.
response = bedrock_client.converse(
    modelId=MODEL_ID,
    messages=messages,
    inferenceConfig=inference_config,
    toolConfig={'ttols':[calculator_tool]}
)

In [ ]:
response["output"]["message"]["content"][0]

## Run the Tool

In [ ]:
tool_requests = response["output"]["message"]["content"][0]["toolUse"]

tool_name = tool_request["name"]
tool_inputs = tool_requests["input"]

print("The Tool Name Claude Wants To Call:", tool_name)
print("The Inputs Claude Wants To Call It With:", tool_inputs)

In [ ]:
operation = tool_inputs["operation"]
operand1 = tool_inputs["operand1"]
operand2 = tool_inputs["operand2"]

result = calculator(operation,operand1,operand2)
print("RESULT IS", result)

## Put it All Together

In [ ]:
def prompt_claude_with_calculator(prompt):
    messages=[{
        "role": "user",
        "content": [{"text": "Multiply 1982982 by 2390831. Only respond with the result"}]
    }]

    inference_config={"maxTokens":400}

    # Send the message.
    response = bedrock_client.converse(
        modelId=MODEL_ID,
        messages=messages,
        inferenceConfig=inference_config,
        toolConfig={'ttols':[calculator_tool]}
    )
    stop_reason = response["stopReason"]

    if stop_reason == "tool_use":
        tool_requests = response["output"]["message"]["content"][-1]
        if response["stopReason"] == "tool_use":
            tool = tool_requests["toolUse"]

            if tool["name"] == "calculator":
                result = calculator(tool["input"]["operation"],tool["input"]["operand1"],tool["input"]["operand2"])
                print("Calculation result is:", result)
    elif stop_reason == "end_turn":
        print("result is:", response["output"]["message"]["content"][0]["text"])

In [ ]:
prompt_claude_with_calculator("what is four times nine")

In [ ]:
prompt_claude_with_calculator("who won the 2022 world series")